In [1]:
import caldis, sys
print("python :", sys.executable)
print("caldis :", caldis.__file__, getattr(caldis, "__version__", "?"))

python : d:\MyWork\Travail\2026\EnergySimulator\caldis-dev\.venv-dev\Scripts\python.exe
caldis : D:\MyWork\Travail\2026\EnergySimulator\caldis-dev\caldis\__init__.py 0.3.1


# Caldis — A Heat Exchanger on its own

After the first example (a compressor), here is the second recurring building block:
a **counter-flow heat exchanger**, modelled alone between four boundaries — a source
and a sink on each side. Hot water enters on the hot side, cold water on the cold
side; the exchanger transfers heat from one stream to the other.

The goal of this notebook is to show two things the compressor example did not need:
- **`add_constraint`** — imposing an *arbitrary relation* (here the exchanger duty
  `Qdot`), which `fix_port` cannot express because it spans several ports;
- **`set_internal_start`** — seeding a component's *internal* unknown (here the
  conductance `UA`), which has no port.

Key point on degrees of freedom: a `SizingHX` with both inlets fully specified is
**one equation short** — the two streams are coupled only by energy conservation,
which leaves one outlet enthalpy free. You close it with **one** thermal condition:
a duty, a pinch, or an outlet temperature. Here we impose the **duty**.

In [2]:
# --- imports ---
import numpy as np

import caldis
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components.HEX.heat_exchanger import SizingHX
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve
from caldis.solvers.autostart import autostart, fix_port

## 1. Build & connect

A `SizingHX` has four ports: `hot_in`, `hot_out`, `cold_in`, `cold_out`. We wire a
`FluidBoundary` to each:
- **`hot_src`** imposes the hot inlet (`p`, `T`, `mdot`), **`hot_snk`** just
  terminates the hot outlet (bare, no spec);
- **`cold_src`** imposes the cold inlet, **`cold_snk`** terminates the cold outlet.

The outlet pressures are not imposed: the exchanger carries them (`dp = 0` per side),
so they follow from the inlets. `SizingHX` also takes per-side film coefficients
(`h_hot`, `h_cold`) and an area ratio — these turn the required conductance into a
physical area after the solve.

In [3]:
be = CoolPropBackend("Water")

p_hot,  T_hot_in,  m_hot  = 2.0e5, 90 + 273.15, 1.0     # hot water in:  2 bar, 90 C, 1 kg/s
p_cold, T_cold_in, m_cold = 2.0e5, 15 + 273.15, 1.0     # cold water in: 2 bar, 15 C, 1 kg/s
Q_target = 1.0e5                                        # duty to transfer: 100 kW

hex = SizingHX("hex", backend_hot=be, backend_cold=be,
               h_hot=500.0, h_cold=500.0, area_ratio_cold_to_hot=1.0)

hot_src  = FluidBoundary("hot_src",  p=p_hot,  T=T_hot_in,  mdot=-m_hot,  backend=be)
hot_snk  = FluidBoundary("hot_snk",  backend=be)          # bare
cold_src = FluidBoundary("cold_src", p=p_cold, T=T_cold_in, mdot=-m_cold, backend=be)
cold_snk = FluidBoundary("cold_snk", backend=be)          # bare

sys_ = System("hex_demo")
sys_.add(hex, hot_src, hot_snk, cold_src, cold_snk)
sys_.connect(hot_src.port,  hex.hot_in)
sys_.connect(hex.hot_out,   hot_snk.port)
sys_.connect(cold_src.port, hex.cold_in)
sys_.connect(hex.cold_out,  cold_snk.port)

Connection(hex.cold_out <-> cold_snk.port)

## 2. Close the exchanger — `add_constraint`

The missing condition is the exchanger **duty**. That is a relation *across ports*
(`mdot · Δh` on the hot side), not a single port value, so `fix_port` does not fit —
this is exactly what **`add_constraint`** is for: you pass a function that must equal
zero, with a name for diagnostics.

```
Qdot_hot = hot_in.mdot * (hot_in.h - hot_out.h)   # heat released by the hot stream
```

Alternatives for the same closure, depending on what you know: a pinch
(`add_constraint(lambda: hex.hot_out.T - hex.cold_in.T - 5)`), or simply fixing an
outlet temperature (`fix_port(sys_, hex.hot_out, T=...)`). Pick exactly one.

In [4]:
sys_.add_constraint(
    lambda: hex.hot_in.mdot * (hex.hot_in.h - hex.hot_out.h) - Q_target,
    "duty",
)

<function __main__.<lambda>()>

## 3. Seed the solver

Two things the propagation cannot derive on its own:
- the **outlet temperatures** must be *distinct* from the inlets, otherwise the
  log-mean temperature difference is degenerate — a `set_start_port` hint each;
- the exchanger's **internal** conductance `UA` has no port — `set_internal_start`
  (a rough `UA ≈ Q / ΔT_lm`).

Then `autostart` fills pressures and flows from the boundaries.

In [5]:
# distinct outlet temperatures (nonlinear) -> set_start_port, one per port
sys_.set_start_port(hex.hot_out,  p=p_hot,  T=70 + 273.15)    # hot stream cooled
sys_.set_start_port(hex.cold_out, p=p_cold, T=35 + 273.15)    # cold stream warmed

# internal unknown (no port): a rough conductance  UA ~ Q / dT_lm  (dT_lm ~ 50 K)
hex.set_internal_start("UA", 2.0e3)

autostart(sys_, verbose=True)

autostart: 14 propagated, 0 pressure fallbacks, 0 port-quantities left at default


## 4. Solve

`solve()` is Newton-first (direct Newton from the seed, homotopy as fallback) and
runs a physical-validity check afterwards. If the duty were too large for the
available temperature span, the streams would cross and the check would warn — here
it stays silent.

In [6]:
sol = solve(sys_)

✓ DOF: square (26 unknowns)
✓ start point: evaluable
  direct Newton: ok (5 it, |F_scaled|=1.8e-10)
✓ SOLVED (|F_scaled|=1.8e-10)


## 5. Read the results

The exchanger exposes derived outputs after the solve: `hex.Qdot` (duty), `hex.UA`,
`hex.A_total` (area, from the film coefficients), `hex.DTLM`. Check that the imposed
duty came out, and that both streams moved the right way (hot cooled, cold warmed).

In [7]:
print(f"hot  : {hex.hot_in.T_C:6.1f} -> {hex.hot_out.T_C:6.1f} degC   ({hex.hot_in.mdot:.2f} kg/s)")
print(f"cold : {hex.cold_in.T_C:6.1f} -> {hex.cold_out.T_C:6.1f} degC   ({hex.cold_in.mdot:.2f} kg/s)")
print(f"duty Qdot = {hex.Qdot/1e3:.1f} kW   (target {Q_target/1e3:.1f} kW)")
print(f"UA = {hex.UA:.0f} W/K   A_total = {hex.A_total:.2f} m2   DTLM = {hex.DTLM:.1f} K")

sys_.port_table()

hot  :   90.0 ->   66.2 degC   (1.00 kg/s)
cold :   15.0 ->   38.9 degC   (1.00 kg/s)
duty Qdot = 100.0 kW   (target 100.0 kW)
UA = 1956 W/K   A_total = 15.65 m2   DTLM = 51.1 K


p_bar           T        T_C      h_kJkg   s_kJkgK    x  \
component port                                                                
hex       hot_in      2.0  363.150000  90.000000  377.139850  1.192771 -1.0   
          hot_out     2.0  339.315124  66.165124  277.139850  0.907957 -1.0   
          cold_in     2.0  288.150000  15.000000   63.171297  0.224433 -1.0   
          cold_out    2.0  312.065459  38.915459  163.171297  0.557828 -1.0   
hot_src   port        2.0  363.150000  90.000000  377.139850  1.192771 -1.0   
hot_snk   port        2.0  339.315124  66.165124  277.139850  0.907957 -1.0   
cold_src  port        2.0  288.150000  15.000000   63.171297  0.224433 -1.0   
cold_snk  port        2.0  312.065459  38.915459  163.171297  0.557828 -1.0   

                           rho  mdot  
component port                        
hex       hot_in    965.354766   1.0  
          hot_out   979.957355  -1.0  
          cold_in   999.148689   1.0  
          cold_out  992.670119  -1.0  
hot_src   port      965.354766  -1.0  
hot_snk   port      979.957355   1.0  
cold_src  port      999.148689  -1.0  
cold_snk  port      992.670119   1.0

## 6. Explore

`structure()` lists the ports and the internal variables (`UA`, `A_hot`) — the latter
being exactly what `set_internal_start` seeds. `describe()` shows every value after
the solve.

In [8]:
hex.structure()
print()
hex.describe();

SizingHX 'hex'
  var    UA
  var    A_hot
  port   hot_in  (Water)   -> .p .mdot .h
  port   hot_out  (Water)   -> .p .mdot .h
  port   cold_in  (Water)   -> .p .mdot .h
  port   cold_out  (Water)   -> .p .mdot .h
  outputs ['A_cold', 'A_total', 'DTLM', 'Qdot', 'U_overall_cold', 'U_overall_hot']

=== SizingHX('hex') ===
  variables:
    UA               = 1956
    A_hot            = 7.82399
  ports (p [bar], mdot [kg/s], T [°C]):
    hot_in     p=2.000  mdot=+1  h=377.1kJ/kg  T=90.00
    hot_out    p=2.000  mdot=-1  h=277.1kJ/kg  T=66.17
    cold_in    p=2.000  mdot=+1  h=63.2kJ/kg  T=15.00
    cold_out   p=2.000  mdot=-1  h=163.2kJ/kg  T=38.92
  outputs:
    U_overall_hot    = 250
    U_overall_cold   = 250
    A_cold           = 7.82399
    A_total          = 15.648
    Qdot             = 100000
    DTLM             = 51.1248
